# MedRAG baseline benchmark (30 questions)

Answers reviewer 1, major comment 2: "A controlled general biomedical RAG baseline is
missing. The T21 Research Assistant is compared mainly with base/all purpose models."

MedRAG (https://github.com/Teddy-XiongGZ/MedRAG) is a published biomedical RAG toolkit.
Here it runs over the SAME corpus the T21 Research Assistant indexes, with the SAME
generator model, so the comparison isolates the retrieval and answering machinery rather
than confounding it with corpus coverage or model quality.

What is held constant, and what differs:

- Same documents: the 1,594 article-only papers in the live Milvus collection
  (DS_INCLUDE_2026_09_02_DEEPINFRA_2048, 65,677 vectors; fresh round 2026-09-06).
- Same generator: nemotron-3-super via the Ollama Cloud OpenAI-compatible endpoint.
- Same scoring: all six arms are scored together in notebook 7 (six-method
  comparison) with the same sentence-level adapted BERTScore via /embed_batch.
  This notebook only GENERATES the 30 MedRAG answers and saves them; its own
  scoring cells (6 onward) are optional standalone diagnostics.
- Different: MedRAG chunks flat text at about 1000 characters with no section labels,
  retrieves with MedCPT dense search, and has no section prioritisation, no
  cross-encoder rerank and no multi-agent pipeline. That difference is the thing under
  test.

Environment: run this notebook with the `medrag` conda kernel (Python 3.10), not
`dschat`. MedRAG pins openai==0.28.0, which conflicts with the production environment.

    conda run -n medrag python -m ipykernel install --user --name medrag

Prerequisites, both produced by notebook 4 (baseline corpora build):
  1. the chunk corpus at corpus/t21_2026_09_02_articles/chunk/*.jsonl (1,594 papers,
     60,391 chunks)
  2. the MedCPT index at corpus/t21_2026_09_02_articles/index/ncbi/... (about 35
     minutes on CPU to build; cached afterwards, loads in seconds)


## T21 Research Assistant vs MedRAG: what is compared

Held constant in the benchmark (the point of the controlled baseline):

| Aspect | Both arms |
|---|---|
| Corpus | Same 1,594 article-only papers (T21_2026_09_02_ARTICLES build, matching the live collection exactly; earlier runs used the superseded 1,602- and 1,765-paper sets) |
| Generator | nemotron-3-super via the Ollama Cloud OpenAI-compatible endpoint, temperature 0.0 |
| Questions | Same 30 positive-control questions with expert reference answers |
| Scoring | Same sentence-level adapted BERTScore + semantic similarity, computed through the production /embed_batch endpoint |

What differs (the thing under test):

| Dimension | T21 Research Assistant | MedRAG baseline |
|---|---|---|
| Chunking | 1,200 chars, 200 overlap, each chunk labeled with its section | ~1,000-char flat chunks, no section labels |
| Embedder | llama-nemotron-embed-vl-1b-v2, 2048-dim, via DeepInfra | MedCPT query/article encoders (768-dim biomedical dual encoder) |
| Vector store | Milvus IVF_FLAT, inner product, nprobe 64 | FAISS index built by MedRAG |
| Section awareness | Per-section retrieval over Results, Discussion, Methods, Conclusion (12 candidates each); Abstract and Introduction excluded entirely | One flat dense search over all text |
| Reranking | Cross-encoder (llama-nemotron-rerank-vl-1b-v2) blended with vector score at alpha 0.8, then section priority multiplier | None |
| Paper diversity | Max 2 chunks per PMCID | None; one paper can dominate |
| Snippets to LLM | top_k 12 (dynamic boost up to 20 for complex/list questions) | k = 32 |
| Pipeline | 8-node LangGraph: question judge, retrieve, rerank, synthesize, raw evidence, safety guard, finalize, judge-improve | Single retrieve-then-answer step |
| LLM calls per question | 2 to 4 (synthesis plus judge/improve; pre-flight judge cached) | 1 |
| Question validation | Pre-flight VALID/INVALID classifier; evidence-gap refusal path | None; always answers |
| Output format | Structured Summary (150-word cap) + Key Evidence + Reasoning; every sentence cited as paper plus section, INCLUDE badges | Free text, ~250 words, no length cap, cites by paper only |
| Safety layer | Prescription/directive language filter | None |


In [1]:
# ---- Cell 1: configuration ----
import json, os, re, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

NB_DIR   = Path.cwd()
ANALYSIS = NB_DIR.parent if NB_DIR.name == "nbs" else NB_DIR
BENCH    = ANALYSIS / "benchmarks"
SRV      = ANALYSIS.parent / "srv"
OUT_DIR  = ANALYSIS / "output" / "medrag_benchmark"; OUT_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS_XLSX = ANALYSIS / "input" / "Revised_DownSyndrome_60_Questions_Final_linked.xlsx"
T21_EMBED_BATCH_URL = "http://localhost:8698/embed_batch"   # scoring embedder
T21_ANSWER_URL      = "http://localhost:8698/answer"        # optional paired arm

# Which MedRAG corpus to run against. "T21_2026_09_02_ARTICLES" is the 1,594-paper
# article-only corpus built by the baseline-corpora notebook (fresh round 2026-09-06),
# matching the live collection exactly. Auto-registered by the adapter.
MEDRAG_CORPUS = "T21_2026_09_02_ARTICLES"

RETRIEVE_K   = 32      # MedRAG default snippet count
QUERY_T21    = False   # kept False: T21 answers come from notebook 3; the paired
                       # T21 arm cell was removed - notebook 7 does the comparison
T21_TOP_K    = 12      # matches DEFAULT_TOPK in docker-compose

sys.path.insert(0, str(BENCH))
print("benchmarks dir :", BENCH)
print("questions file :", QUESTIONS_XLSX.name, "| exists:", QUESTIONS_XLSX.exists())
print("output dir     :", OUT_DIR)

# ── Live collection gate (fresh-results round 2026-09-06) ─────────────────────
EXPECTED_COLLECTION = "DS_INCLUDE_2026_09_02_DEEPINFRA_2048"
EXPECTED_VECTORS, EXPECTED_DOCS = 65_677, 1_594
_hj = requests.get("http://localhost:8698/health", timeout=30).json()
_hc = _hj["config"]["milvus_collection"]
_hv = int(str(_hj["database"]["total_vectors"]).replace(",", ""))
_hd = int(str(_hj["database"]["estimated_documents"]).replace(",", ""))
assert (_hc, _hv, _hd) == (EXPECTED_COLLECTION, EXPECTED_VECTORS, EXPECTED_DOCS), (
    f"API serves {_hc} ({_hv:,} vectors / {_hd:,} docs); expected "
    f"{EXPECTED_COLLECTION} ({EXPECTED_VECTORS:,} / {EXPECTED_DOCS:,})")
print(f"Collection gate OK: {_hc} ({_hv:,} vectors, {_hd:,} documents)")



benchmarks dir : /path/to/project/T21_research_assistant_v7/analysis/benchmarks
questions file : Revised_DownSyndrome_60_Questions_Final_linked.xlsx | exists: True
output dir     : /path/to/project/T21_research_assistant_v7/analysis/output/medrag_benchmark


Collection gate OK: DS_INCLUDE_2026_09_02_DEEPINFRA_2048 (65,677 vectors, 1,594 documents)


## Cell 2: load the 30 positive-control questions

The 60-question set contains 30 positive controls (answerable from the DS literature,
each with an expert reference answer and supporting PMIDs) and 30 negative controls
(deliberately out of scope). BERTScore is computed on the positive controls only,
because a negative control has no reference answer to score against; the negatives test
refusal behaviour and are not scored in this benchmark.

Using 30 rather than the original 12 also addresses reviewer 1, major comment 4 on
benchmark size.


In [2]:
# ---- Cell 2: questions ----
df_q = pd.read_excel(QUESTIONS_XLSX, sheet_name="Questions_for_testing")
df_q.columns = [c.strip() for c in df_q.columns]
pos = df_q[df_q["CTRL_Type"].str.strip().str.lower() == "positive"].copy().reset_index(drop=True)
pos = pos.rename(columns={"Original question": "question", "Answer": "ground_truth"})
pos["question"] = pos["question"].astype(str).str.strip()
pos["ground_truth"] = pos["ground_truth"].astype(str).str.strip()

print(f"positive controls: {len(pos)}")
print(f"negative controls: {(df_q['CTRL_Type'].str.strip().str.lower() == 'negative').sum()} (not scored here)")
assert len(pos) == 30, f"expected 30 positive controls, found {len(pos)}"
pos[["SLNo", "question"]].head(5)


positive controls: 30
negative controls: 30 (not scored here)


,SLNo,question
0,Q1,Which clinical conditions are people with Down...
1,Q2,Which clinical conditions are people with Down...
2,Q3,Why do people with Down syndrome tend to have ...
3,Q4,Why are people with Down syndrome at high risk...
4,Q5,Do individuals with Down syndrome have overact...


## Cell 3: build the MedRAG retriever

First call constructs the MedCPT index over the T21 corpus and takes roughly 35 minutes
on CPU; afterwards it loads from disk in seconds. The generator is pointed at
nemotron-3-super, and MedRAG's multiple-choice templates are replaced with its
free-text ones, since the benchmark questions are open ended.


In [3]:
# ---- Cell 3: MedRAG ----
from medrag_adapter import build_medrag, answer_freetext

t0 = time.time()
medrag = build_medrag(corpus=MEDRAG_CORPUS, retriever="MedCPT", provider="ollama_remote")
print(f"MedRAG ready in {time.time()-t0:.0f}s")
print("  generator :", medrag.model)
print("  retriever : MedCPT (ncbi/MedCPT-Query-Encoder)")
print(f"  corpus    : {MEDRAG_CORPUS} (flat ~1000-char chunks)")


/home/user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/user/anaconda3/envs/medrag/lib/python3.10/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/user/anaconda3/envs/medrag/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'cached_download' (from 'huggingface_hub.file_download') is deprecated and will be removed from version '0.26'. Use `hf_hub_download` instead.
  warnings.warn(warning_message, FutureWarning)


No sentence-transformers model found with name /home/user/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.


MedRAG ready in 3s
  generator : nemotron-3-super
  retriever : MedCPT (ncbi/MedCPT-Query-Encoder)
  corpus    : T21_2026_09_02_ARTICLES (flat ~1000-char chunks)


## Cell 4: run the 30 questions

Checkpointed to `medrag_answers_t21_2026_09_02_articles.json` (corpus-specific, so answers from one corpus are never reused for another), so an interrupted run resumes instead of
repeating paid calls. Retrieved snippet ids are stored alongside each answer so the
grounding of any answer can be audited later.


In [4]:
# ---- Cell 4: MedRAG answers ----
# Answers are corpus-specific: a checkpoint from one corpus must never satisfy a run
# against another, or the benchmark silently reports stale answers.
ANS_PATH = OUT_DIR / f"medrag_answers_{MEDRAG_CORPUS.lower()}.json"
answers = json.loads(ANS_PATH.read_text()) if ANS_PATH.exists() else {}

for i, row in pos.iterrows():
    key = str(row["SLNo"])
    if key in answers and answers[key].get("answer"):
        continue
    q = row["question"]
    t0 = time.time()
    try:
        text, snips = answer_freetext(medrag, q, k=RETRIEVE_K)
        answers[key] = {
            "question": q,
            "answer": text,
            "secs": round(time.time() - t0, 1),
            "snippet_ids": [s.get("id", "") for s in snips],
            "snippet_papers": sorted({s.get("title", "") for s in snips}),
        }
    except Exception as exc:
        answers[key] = {"question": q, "answer": "", "error": str(exc)[:200]}
        print(f"  [{key}] ERROR {str(exc)[:110]}")
    ANS_PATH.write_text(json.dumps(answers, indent=1))
    print(f"  [{i+1}/{len(pos)}] {answers[key].get('secs', '-')}s  {q[:62]}")

pos["medrag_answer"] = [answers.get(str(s), {}).get("answer", "") for s in pos["SLNo"]]
n_ok = int((pos["medrag_answer"].str.len() > 0).sum())
print(f"\nanswered: {n_ok}/{len(pos)}")
print(f"mean answer length: {pos['medrag_answer'].str.split().str.len().mean():.0f} words")


  [12/30] 10.9s  What type of cancer are individuals with Down Syndrome less pr

answered: 30/30
mean answer length: 280 words


## Cell 5: save results for the six-method comparison

Writes `medrag_benchmark_results.csv` (question, ground truth, MedRAG answer) the
moment generation completes. Notebook 7 reads this file and computes all scores
itself, so in the generation-only flow THIS IS THE LAST CELL TO RUN; everything
below is optional standalone diagnostics.


In [5]:
# ---- Save results for the six-method comparison (notebook 7) ----
# Generation-only flow: notebook 7 recomputes all scores itself from the raw
# answers, so this CSV (question, ground_truth, medrag_answer) is all it needs.
# The scoring cells below are OPTIONAL standalone diagnostics.
pos.to_csv(OUT_DIR / "medrag_benchmark_results.csv", index=False)
print(f"saved -> {OUT_DIR / 'medrag_benchmark_results.csv'}  ({len(pos)} rows)")


saved -> /path/to/project/T21_research_assistant_v7/analysis/output/medrag_benchmark/medrag_benchmark_results.csv  (30 rows)


## Cell 6 (optional): sentence-level adapted BERTScore

Identical to notebook 3, deliberately: same sentence splitter (including the semicolon
rule for the key-point style of the expert answers), same embedder (the production
/embed_batch endpoint), and the same precision, recall and F definitions. Reusing the
implementation rather than rewriting it is what makes the MedRAG numbers comparable to
the numbers already reported for GPT, Claude and Nemotron.

Note the terminology the reviewer asked for: this is a sentence-level ADAPTED BERTScore,
not the published BERTScore metric. It compares sentence embeddings rather than token
embeddings.


In [6]:
# ---- Cell 6: scoring ----
_SENT_RE_SC = re.compile(r"(?<=[.!?])\s+")
_SEMI_RE_SC = re.compile(r"\s*;\s*")

_ERR_MARKER_RE_SC = re.compile(r"\[[A-Z][A-Z0-9_]*_ERROR\]|too many concurrent", re.I)

def _is_invalid_answer_sc(text) -> bool:
    """Empty or API-error answers are EXCLUDED from scoring (NaN), never scored as 0.
    Genuine short model outputs ("I cannot answer.") are NOT invalid and stay in."""
    t = str(text or "").strip()
    return (not t) or t.lower() == "nan" or bool(_ERR_MARKER_RE_SC.search(t))

def _split_sents_sc(text):
    raw = _SENT_RE_SC.split(str(text or "").strip())
    result = []
    for chunk in raw:
        chunk = chunk.strip()
        if not chunk:
            continue
        if ";" in chunk and not re.search(r"[.!?]\s*$", chunk):
            result.extend([p.strip() for p in _SEMI_RE_SC.split(chunk) if p.strip()])
        else:
            result.append(chunk)
    return result

def _embed_batch_sc(texts):
    r = requests.post(T21_EMBED_BATCH_URL, json={"texts": texts, "mode": "document"}, timeout=180)
    r.raise_for_status()
    return np.array(r.json()["embeddings"], dtype=np.float32)

def _bertscore_sc(reference, candidate):
    # NaN (excluded from means), not 0: empty/unsplittable input means "cannot score".
    _default = {"bertscore_p": float("nan"), "bertscore_r": float("nan"),
                "bertscore_f": float("nan")}
    ref_sents, cand_sents = _split_sents_sc(reference), _split_sents_sc(candidate)
    if not ref_sents or not cand_sents:
        return _default
    try:
        vecs = _embed_batch_sc(ref_sents + cand_sents)
        vecs = vecs / np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-9)
        sim = vecs[:len(ref_sents)] @ vecs[len(ref_sents):].T
        recall = float(sim.max(axis=1).mean())
        precision = float(sim.max(axis=0).mean())
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall > 0 else 0.0
        return {"bertscore_p": round(precision, 4), "bertscore_r": round(recall, 4),
                "bertscore_f": round(f1, 4)}
    except Exception as exc:
        print(f"  [bertscore_err] {exc}")
        return {k: float("nan") for k in _default}

def _semantic_sim(reference, candidate):
    """Document-level cosine similarity, the second metric used in notebook 3."""
    if not str(reference).strip() or not str(candidate).strip():
        return float("nan")
    v = _embed_batch_sc([str(reference), str(candidate)])
    v = v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-9)
    return float(round(v[0] @ v[1], 4))

# Derive the arms from what is actually present, so scoring never depends on
# whether an earlier cell was run, skipped, or interrupted.
arms = [(a, col) for a, col in [("medrag", "medrag_answer"), ("t21", "t21_answer")]
        if col in pos.columns and pos[col].astype(str).str.strip().ne("").any()]
print("scoring arms:", [a for a, _ in arms])
if not any(a == "t21" for a, _ in arms):
    print("  note: T21 answers come from notebook 3; notebook 7 does the comparison")
for arm, col in arms:
    print(f"scoring {arm} ...")
    res = [({"bertscore_p": float("nan"), "bertscore_r": float("nan"), "bertscore_f": float("nan")}
            if _is_invalid_answer_sc(r[col])
            else _bertscore_sc(r["ground_truth"], r[col]))
           for _, r in pos.iterrows()]
    for k in ("bertscore_p", "bertscore_r", "bertscore_f"):
        pos[f"{arm}_{k}"] = [x[k] for x in res]
    pos[f"{arm}_semantic_sim"] = [(float("nan") if _is_invalid_answer_sc(r[col])
                                   else _semantic_sim(r["ground_truth"], r[col]))
                                  for _, r in pos.iterrows()]
    _n_bad = int(pos[col].map(_is_invalid_answer_sc).sum())
    if _n_bad:
        print(f"  [EXCLUDED] {arm}: {_n_bad} empty/API-error answer(s) -> NaN (dropped from means)")
print("done")


scoring arms: ['medrag']
  note: T21 answers come from notebook 3; notebook 7 does the comparison
scoring medrag ...


done


## Cell 6b (optional): size-matched scoring

MedRAG answers average about 250 words; the T21 summary is capped at 150 and the expert
reference answers are shorter still. Length alone moves these metrics: a longer answer
tends to raise recall (more of the reference is covered somewhere) and depress precision
(more sentences with no counterpart in the reference), independent of retrieval quality.

This repeats the scoring on length-matched answers, using the same symmetric procedure
as notebook 5: embed the reference and every candidate sentence, rank the candidate
sentences by cosine similarity to the reference, then greedily keep the best ones in
their original order until the reference word count is reached. Applying the identical
rule to every arm means no arm gains an advantage from verbosity.

Report both. The full-length numbers describe the systems as a user experiences them;
the size-matched numbers isolate content quality.


In [7]:
# ---- Cell 6b: size-matched scoring ----
def _size_match(reference, candidate):
    """Trim candidate to the reference word count, keeping its most relevant sentences."""
    ref, cand = str(reference or "").strip(), str(candidate or "").strip()
    if not ref or not cand:
        return cand
    target = len(ref.split())
    sents = _split_sents_sc(cand)
    if len(sents) <= 1 or len(cand.split()) <= target:
        return cand
    vecs = _embed_batch_sc([ref] + sents)
    vecs = vecs / np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-9)
    sims = vecs[1:] @ vecs[0]
    keep, used = set(), 0
    for idx in np.argsort(-sims):                 # best first
        w = len(sents[idx].split())
        if used and used + w > target:
            continue
        keep.add(int(idx)); used += w
        if used >= target:
            break
    if not keep:
        keep = {int(np.argmax(sims))}
    return " ".join(sents[i] for i in sorted(keep))   # original order preserved

for arm, col in arms:
    print(f"size-matching {arm} ...")
    pos[f"{arm}_answer_sm"] = [_size_match(r["ground_truth"], r[col]) for _, r in pos.iterrows()]
    res = [_bertscore_sc(r["ground_truth"], r[f"{arm}_answer_sm"]) for _, r in pos.iterrows()]
    for k in ("bertscore_p", "bertscore_r", "bertscore_f"):
        pos[f"{arm}_sm_{k}"] = [x[k] for x in res]

print()
for arm, col in arms:
    full = pos[f"{arm}_bertscore_f"].mean()
    sm = pos[f"{arm}_sm_bertscore_f"].mean()
    wc_full = pos[col].str.split().str.len().mean()
    wc_sm = pos[f"{arm}_answer_sm"].str.split().str.len().mean()
    print(f"{arm:8s} BERTScore F  full {full:.4f} ({wc_full:.0f} words) -> "
          f"size-matched {sm:.4f} ({wc_sm:.0f} words)")
print(f"reference answers average {pos['ground_truth'].str.split().str.len().mean():.0f} words")


size-matching medrag ...



medrag   BERTScore F  full 0.5646 (280 words) -> size-matched 0.5934 (92 words)
reference answers average 76 words


## Cell 7 (optional): results

Reports the mean of each metric per arm, and, when both arms ran, a paired Wilcoxon
signed-rank test on BERTScore F. Paired is the right test here because both arms answer
the identical 30 questions.

Read the p value with care. With n=30 a null result means "no difference detected at
this sample size", not "the systems are equivalent", which is exactly the inference
error reviewer 1 flagged in major comment 4. Report the effect size alongside it.


In [8]:
# ---- Cell 7: results ----
metrics = ["bertscore_p", "bertscore_r", "bertscore_f", "semantic_sim"]
rows = {}
for arm, _ in arms:
    rows[arm] = [pos[f"{arm}_{m}"].mean() for m in metrics]
    sm_col = f"{arm}_sm_bertscore_f"
    if sm_col in pos.columns:
        rows[f"{arm} (size-matched F)"] = [np.nan, np.nan, pos[sm_col].mean(), np.nan]
summary = pd.DataFrame(rows, index=metrics).round(4)
print("mean scores over 30 positive-control questions")
print(summary.to_string())

if len(arms) == 2:
    from scipy.stats import wilcoxon
    a = pos["t21_bertscore_f"].astype(float)
    b = pos["medrag_bertscore_f"].astype(float)
    ok = a.notna() & b.notna()
    stat, p = wilcoxon(a[ok], b[ok])
    diff = float((a[ok] - b[ok]).mean())
    print(f"\npaired Wilcoxon on BERTScore F (n={int(ok.sum())})")
    print(f"  mean difference (T21 minus MedRAG): {diff:+.4f}")
    print(f"  statistic={stat:.1f}  p={p:.4f}")
    print("  note: a null result at n=30 means no difference detected, not equivalence")

pos.to_csv(OUT_DIR / "medrag_benchmark_results.csv", index=False)
summary.to_csv(OUT_DIR / "medrag_benchmark_summary.csv")
print(f"\nsaved to {OUT_DIR}")


mean scores over 30 positive-control questions
              medrag  medrag (size-matched F)
bertscore_p   0.5405                      NaN
bertscore_r   0.6021                      NaN
bertscore_f   0.5646                   0.5934
semantic_sim  0.6789                      NaN

saved to /path/to/project/T21_research_assistant_v7/analysis/output/medrag_benchmark


## Cell 8 (optional): per-question view and grounding audit

Useful for the response to reviewers: it shows where the arms diverge, and lets any
MedRAG answer be traced back to the papers it retrieved, the same way T21 answers carry
citations. One qualitative difference worth noting in the manuscript is that MedRAG
cites by paper only, whereas T21 cites paper plus section.


In [9]:
# ---- Cell 8: per-question detail ----
cols = ["SLNo", "question"] + [f"{a}_bertscore_f" for a, _ in arms]
detail = pos[cols].copy()
if len(arms) == 2:
    detail["delta_f"] = (detail["t21_bertscore_f"] - detail["medrag_bertscore_f"]).round(4)
    detail = detail.sort_values("delta_f")
print(detail.to_string(index=False, max_colwidth=58))

# Which questions does the baseline handle worst? This is the useful part for the
# manuscript: an aggregate mean says a difference exists, the failure pattern says why.
print("\nweakest 5 questions for medrag (BERTScore F)")
for _, r in pos.nsmallest(5, "medrag_bertscore_f").iterrows():
    print(f"  {r['medrag_bertscore_f']:.3f}  {str(r['question'])[:76]}")
print("\nMeasured 2026-09-02: the worst case was a negative-polarity question")
print("(which cancers are people with DS LESS prone to), followed by specific")
print("mechanistic queries. Both are cases where the answer sits in Results or")
print("Methods rather than background prose, which is where section-aware retrieval")
print("is expected to help. Check whether T21 scores higher on these same items.")

# grounding audit for one question
k0 = str(pos.iloc[0]["SLNo"])
rec = answers.get(k0, {})
print(f"\nGROUNDING AUDIT for SLNo {k0}")
print("  question :", rec.get("question", "")[:100])
print("  papers   :", ", ".join(rec.get("snippet_papers", [])[:8]))
print("  answer   :", rec.get("answer", "")[:400])


SLNo                                                   question  medrag_bertscore_f
  Q1 Which clinical conditions are people with Down syndrome...              0.5635
  Q2 Which clinical conditions are people with Down syndrome...              0.6533
  Q3 Why do people with Down syndrome tend to have hyperacti...              0.5279
  Q4 Why are people with Down syndrome at high risk of devel...              0.5738
  Q5 Do individuals with Down syndrome have overactive thyro...              0.6188
  Q6 Which interferon receptors are triplicated in Down synd...              0.7376
  Q7          How is Stat1 signaling affected in Down syndrome?              0.5422
  Q8 Which circulating immune cell is consistently depleted ...              0.3401
  Q9 What are some advantages and caveats of the Ts65Dn and ...              0.5319
 Q10 Are individuals with Down syndrome more prone to high b...              0.7303
 Q11   What neurodegeneration is associated with Down syndrome?             

---
## Notes for the manuscript

Wording. Describe this arm as "MedRAG, run over the same corpus and generator", not as
"MedRAG" unqualified. MedRAG as published retrieves from PubMed, StatPearls, Textbooks
or Wikipedia; here it retrieves from the T21 corpus, which is a deliberate modification
so the comparison is controlled.

Adaptations made, all documented in analysis/benchmarks/medrag_adapter.py:
  1. MedRAG's default prompts are multiple-choice ("output json with answer_choice") and
     were replaced with its own free-text templates, since the benchmark is open ended.
  2. MedRAG sets context-length attributes only for model names containing gpt-3.5 or
     gpt-4, so they are set explicitly for nemotron-3-super.
  3. The generator is reached through MedRAG's OpenAI path with a custom api_base.

What this arm does and does not show. It compares whole retrieval pipelines: MedCPT
dense retrieval over flat chunks against section-aware retrieval with reranking. It does
not isolate section prioritisation alone, because the embedding models also differ. A
tighter ablation, the T21 pipeline with section priorities flattened, is supported by
sections_override in the production code and would isolate that single variable.

Reviewer comment 4 (benchmark size) is partly addressed by moving from 12 to 30 positive
controls. The concern about tuning and evaluation sharing questions still needs an
answer: state explicitly that the alpha and section-condition sweep
(1_alpha_parameter_rerank_optimization.ipynb) used the same 30 positive controls,
and address the tuning/evaluation overlap in the response letter.
